# 03 Backtest and Evaluation

## A. 本步目标
运行滚动、月度调仓的 simple backtest，并比较历史表现、drawdown、turnover、集中度与公开披露口径下的 carbon intensity。

## B. 概念解释
Look-ahead bias 指在决策时使用未来信息。本回测在日期 t 只用 t 之前的窗口估计。Turnover 是相邻调仓权重变化的一半绝对值之和，不是完整交易成本。

## C. 本步默认值
使用 252 日滚动窗口、月度调仓和零交易成本。零成本是明显的简化假设。

In [ ]:
from pathlib import Path
import sys, yaml, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from carbon_portfolio.data import download_adjusted_close, calculate_returns
from carbon_portfolio.carbon import load_carbon_intensity
from carbon_portfolio.backtest import run_backtest
from carbon_portfolio.metrics import performance_summary, turnover, concentration_hhi
from carbon_portfolio.visualization import plot_cumulative_returns, plot_drawdowns
cfg = yaml.safe_load((ROOT / 'config/settings.yaml').read_text())
prices = download_adjusted_close(cfg['data']['tickers'], cfg['data']['start_date'], cfg['data']['end_date'], ROOT / cfg['data']['cache_path'])
returns = calculate_returns(prices)
scores = load_carbon_intensity(ROOT / 'data/carbon_intensity.csv', returns.columns)
result = run_backtest(returns, scores, lookback_days=cfg['backtest']['lookback_days'], rebalance_frequency=cfg['backtest']['rebalance_frequency'], risk_aversion=cfg['optimization']['risk_aversion'], max_weight=cfg['optimization']['max_weight'], carbon_budget_ratio=cfg['optimization']['carbon_budget_ratio'], transaction_cost_bps=cfg['backtest']['transaction_cost_bps'])
summary = performance_summary(result.portfolio_returns, cfg['backtest']['risk_free_rate'])
summary['average_turnover'] = pd.Series({name: turnover(w).mean() for name, w in result.rebalance_weights.items()})
summary['average_hhi'] = pd.Series({name: concentration_hhi(w).mean() for name, w in result.rebalance_weights.items()})
summary

In [ ]:
plot_cumulative_returns(result.portfolio_returns)
plot_drawdowns(result.portfolio_returns)
result.diagnostics.tail()

## Sensitivity analysis
下面的参数网格用于展示结论对 carbon budget 和 risk aversion 的敏感程度。它不是为了从历史样本中挑选表现最好的参数。

In [ ]:
from carbon_portfolio.sensitivity import run_sensitivity_analysis
sensitivity = run_sensitivity_analysis(returns, scores, cfg['sensitivity']['carbon_budget_ratios'], cfg['sensitivity']['risk_aversion_values'], lookback_days=cfg['backtest']['lookback_days'], rebalance_frequency=cfg['backtest']['rebalance_frequency'], max_weight=cfg['optimization']['max_weight'])
sensitivity

## E. 自检
- `estimation_end` 是否严格早于调仓日期？
- 三个组合是否使用相同资产池与调仓日期？
- 是否同时报告失败、集中度和 turnover，而非只展示收益？
- 是否把单一样本 backtest 误称为 alpha 或 downside resilience？不能。

## F. 向 Senior Quant 解释
这是一个使用滚动历史估计的简化回测，用于比较约束机制，而不是验证交易策略。每次调仓只使用此前数据，减少了明显的 look-ahead bias。结果仍受固定资产池、估计误差、参数选择和简化交易成本影响，因此历史 Sharpe 或 drawdown 不能被外推为未来表现。